# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display all available record sets and their fields using their @id
print("Record sets available in the dataset:")
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name}, @id: {field.id}, Data Type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set(s) to work with by @id
# First, gather list of record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

# Example: Load all record sets into separate DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}")
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select the first record set and analyze a numeric field
# First, examine available numeric fields for the first record set
selected_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_record_set_id]
print(f"Analyzing record set: {selected_record_set_id}")

# Show numeric columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields in this record set:", numeric_cols)

# If any numeric field exists, filter and normalize
if numeric_cols:
    numeric_field = numeric_cols[0]
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in "iufc" else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
    print(filtered_df.head())
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())
    
    # Attempt to group by a categorical field
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = None
    if len(categorical_cols) > 0:
        group_field = categorical_cols[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric fields found in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If we also have a group_field, plot boxplot
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric columns available for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`.
- Explored the record set and available fields with their `@id` identifiers.
- Extracted tabular records into pandas DataFrames, filtered, and normalized a numeric column.
- Performed basic EDA and visualized distributions and relationships in the data.

You may continue exploring by selecting additional record sets or fields by their `@id`, or by testing other types of analysis and visualization depending on your research needs.